# Solution: Sklearn Pipeline -- Titanic Extensions

**Exercises from notebook `02_sklearn_pipeline.ipynb`**

Jingxu Li - 320230942071


## Exercise 1: LogTransformer for Fare

Add a LogTransformer custom step that applies log1p to only the fare column before the ColumnTransformer, and show it improves cross-validated accuracy.


In [8]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split

df = pd.read_csv('../dataset/titanic.csv')

class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols if cols else []
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                X[c] = np.log1p(X[c].clip(lower=0))
        return X

feat = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X_all = df[feat]
y_all = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
X = X_all
y = y_all

def make_pre(do_log):
    steps = [
        ('num', Pipeline([
            *([('log', LogTransformer(cols=['fare']))] if do_log else []),
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), ['age', 'sibsp', 'parch', 'fare']),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), ['sex', 'embarked']),
        ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
    ]
    return ColumnTransformer(steps)

results = {}
for tag, use_log in [('baseline', False), ('+log1p(fare)', True)]:
    p = Pipeline([('pre', make_pre(use_log)), ('rf', RandomForestClassifier(n_estimators=100, random_state=42))])
    s = cross_val_score(p, X_tr, y_tr, cv=5, scoring='accuracy')
    results[tag] = s

print(f"{'Config':<20} {'CV acc mean':<12} {'CV acc std':<12}")
print('-' * 44)
for tag, s in results.items():
    print(f"{tag:<20} {s.mean():<12.4f} {s.std():<12.4f}")
d = results['+log1p(fare)'].mean() - results['baseline'].mean()
print(f"\nDelta: {d:+.4f}")


Config               CV acc mean  CV acc std  
--------------------------------------------
baseline             0.7922       0.0258      
+log1p(fare)         0.7950       0.0271      

Delta: +0.0028


## Exercise 2: Extract Deck from Cabin

Extend the Titanic pipeline to also include the deck column (extracted from the cabin number). Handle its high missingness inside the Pipeline.


In [9]:
print(f"deck nulls: {df['deck'].isna().sum()}/{len(df)}")
print(f"deck values: {sorted(df['deck'].dropna().unique())}")

cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X_base = df[cols]
y_tgt = df['survived']
X_w_deck = X_base.assign(deck=df['deck'].fillna('U'))

X_tr, X_te, y_tr, y_te = train_test_split(X_base, y_tgt, test_size=0.2, random_state=42)
X_tr_wd = X_w_deck.loc[X_tr.index]

def build_pipe(inc_deck):
    c = ['sex', 'embarked'] + (['deck'] if inc_deck else [])
    return Pipeline([
        ('pre', ColumnTransformer([
            ('num', Pipeline([('im', SimpleImputer(strategy='median')), ('sc', StandardScaler())]),
             ['age', 'sibsp', 'parch', 'fare']),
            ('cat', Pipeline([('im', SimpleImputer(strategy='most_frequent')),
                              ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), c),
            ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
        ])),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

sn = cross_val_score(build_pipe(False), X_tr, y_tr, cv=5, scoring='accuracy')
sw = cross_val_score(build_pipe(True), X_tr_wd, y_tr, cv=5, scoring='accuracy')

print(f"\n{'variant':<20} {'mean':>8} {'std':>8}")
print('-' * 36)
print(f"{'no deck':<20} {sn.mean():>8.4f} {sn.std():>8.4f}")
print(f"{'+ deck (NaN fill U)':<20} {sw.mean():>8.4f} {sw.std():>8.4f}")
print(f"\ndelta: {sw.mean() - sn.mean():+.4f}")


deck nulls: 688/891
deck values: ['A', 'B', 'C', 'D', 'E', 'F', 'G']

variant                  mean      std
------------------------------------
no deck                0.7922   0.0258
+ deck (NaN fill U)    0.7964   0.0355

delta: +0.0042


## Exercise 3: GridSearchCV for Imputation & Hyperparams

Use GridSearchCV to tune the imputation strategy (median vs most_frequent) for the age column alongside RandomForest hyperparameters. Use Pipeline parameter notation.


In [10]:
from sklearn.model_selection import GridSearchCV

pipe3 = Pipeline([
    ('pre', ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer()), ('sc', StandardScaler())]),
         ['age', 'fare', 'sibsp', 'parch']),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                          ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
         ['sex', 'embarked']),
        ('ord', SimpleImputer(strategy='most_frequent'), ['pclass'])
    ])),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'pre__num__imp__strategy': ['median', 'most_frequent'],
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [5, 8, None]
}

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
gs = GridSearchCV(pipe3, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
gs.fit(X_tr, y_tr)

print('best from CV:')
for k, v in gs.best_params_.items():
    print(f'  {k} = {v}')
print(f'  CV score = {gs.best_score_:.4f}')
print(f'  test acc  = {gs.score(X_te, y_te):.4f}')

results_df = pd.DataFrame(gs.cv_results_)
cols = [c for c in results_df.columns if c.startswith('param_') or c == 'mean_test_score']
print(f'\ntop 3 candidates:')
results_df.sort_values('rank_test_score')[cols].head(3)


best from CV:
  clf__max_depth = 5
  clf__n_estimators = 200
  pre__num__imp__strategy = median
  CV score = 0.8286
  test acc  = 0.8156

top 3 candidates:


,param_clf__max_depth,param_clf__n_estimators,param_pre__num__imp__strategy,mean_test_score
4,5,200,median,0.828612
2,5,100,median,0.827204
0,5,50,median,0.824416


## Exercise 4: Scaler Leakage Demonstration

Demonstrate that fitting the scaler outside the Pipeline before cross_val_score inflates the CV score -- show the numerical difference.


In [11]:
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.model_selection import KFold

Xd, yd = make_classification(n_samples=500, n_features=5, random_state=42)
Xd[-80:] = Xd[-80:] + 10  # last 80 rows shifted

kf = KFold(5, shuffle=False)
sv = SVC(kernel='linear', C=0.01)

p = cross_val_score(Pipeline([('sc', StandardScaler()), ('sv', sv)]), Xd, yd, cv=kf)
Xl = StandardScaler().fit_transform(Xd)
l = cross_val_score(sv, Xl, yd, cv=kf)

print("Fold  pipe   leak   infl")
for i in range(5):
    d = l[i] - p[i]
    print(f" {i+1}   {p[i]:.4f}  {l[i]:.4f}  {d:+.4f}")
print(f"avg   {p.mean():.4f}  {l.mean():.4f}  {l.mean()-p.mean():+.4f}")


Fold  pipe   leak   infl
 1   0.6800  0.7400  +0.0600
 2   0.4600  0.4900  +0.0300
 3   0.5100  0.5800  +0.0700
 4   0.7700  0.8700  +0.1000
 5   0.5300  0.5600  +0.0300
avg   0.5900  0.6480  +0.0580
